In [6]:
import sys
import os
import cv2
import numpy as np


def process(path, out_dir="output"):
    os.makedirs(out_dir, exist_ok=True)
    name = os.path.splitext(os.path.basename(path))[0]

    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(path)

    # 1) 노이즈 제거 -------------------------------------------------
    # 컬러 이미지용 Non-local Means (먼지/그레인 제거, 경계는 최대한 유지)
    den = cv2.fastNlMeansDenoisingColored(img, None, h=6, hColor=6,
                                          templateWindowSize=7, searchWindowSize=21)
    # 약한 가우시안 블러로 잔노이즈 한 번 더 정리
    smooth = cv2.GaussianBlur(den, (5, 5), 1.2)

    # 2) 가우시안 블러 이용한 핵 강조 (Unsharp mask) ------------------
    # 원본 + (원본 - 블러) * amount  -> 경계/핵이 또렷해짐
    blur_big = cv2.GaussianBlur(smooth, (0, 0), sigmaX=8)
    sharp = cv2.addWeighted(smooth, 1.5, blur_big, -0.5, 0)

    # 3) 색 변환: LAB에서 L채널 CLAHE로 대비 올리기 -------------------
    lab = cv2.cvtColor(sharp, cv2.COLOR_BGR2LAB)
    L, A, B = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    L2 = clahe.apply(L)
    enhanced = cv2.cvtColor(cv2.merge([L2, A, B]), cv2.COLOR_LAB2BGR)

    # 핵 구분용 시각화 1: HSV (핵은 채도 높고 명도 낮음)
    hsv = cv2.cvtColor(enhanced, cv2.COLOR_BGR2HSV)

    # 핵 구분용 시각화 2: 어두운 정도를 컬러맵으로
    # 핵은 주변(세포질)보다 훨씬 어두운 작은 덩어리 -> black-hat으로 추출
    gray = cv2.cvtColor(enhanced, cv2.COLOR_BGR2GRAY)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (41, 41))  # 핵보다 큰 커널
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, k)
    blackhat = cv2.normalize(blackhat, None, 0, 255, cv2.NORM_MINMAX)
    heat = cv2.applyColorMap(blackhat, cv2.COLORMAP_JET)

    # 4) 핵 분할 -----------------------------------------------------
    # 핵 = (a) 주변보다 국소적으로 어둡고 (b) 절대 밝기도 낮은 영역
    g = cv2.GaussianBlur(cv2.cvtColor(smooth, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    dark_thr = np.percentile(g, 4)                       # 이미지에서 가장 어두운 4%
    mask = ((g <= dark_thr) & (blackhat > 40)).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,
                            cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE,
                            cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)))

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    result = enhanced.copy()
    count = 0
    for c in contours:
        area = cv2.contourArea(c)
        if area < 150 or area > 6000:  # 이미지 배율에 맞게 조절
            continue
        perim = cv2.arcLength(c, True)
        circ = 4 * np.pi * area / (perim ** 2 + 1e-6)
        if circ < 0.6:  # 둥근 것만
            continue
        count += 1
        cv2.drawContours(result, [c], -1, (0, 255, 0), 2)
        x, y, w, h = cv2.boundingRect(c)
        cv2.putText(result, str(count), (x, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    print(f"[{name}] 검출된 핵: {count}개")

    # 저장
    cv2.imwrite(f"{out_dir}/{name}_1_denoised.png", smooth)
    cv2.imwrite(f"{out_dir}/{name}_2_sharpened.png", sharp)
    cv2.imwrite(f"{out_dir}/{name}_3_enhanced_clahe.png", enhanced)
    cv2.imwrite(f"{out_dir}/{name}_4_hsv.png", hsv)
    cv2.imwrite(f"{out_dir}/{name}_5_heatmap.png", heat)
    cv2.imwrite(f"{out_dir}/{name}_6_mask.png", mask)
    cv2.imwrite(f"{out_dir}/{name}_7_result.png", result)


if __name__ == "__main__":
    paths = [
        "image.png",
        "image2.png",
       
    ]
    for p in paths:
        process(p)

     

[image] 검출된 핵: 20개
[image2] 검출된 핵: 11개


In [ ]:
import sys
import os
import cv2
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# 한글 폰트 (맥: AppleGothic / 윈도우: Malgun Gothic / 없으면 기본값)
for _f in ["AppleGothic", "Malgun Gothic", "NanumGothic"]:
    try:
        matplotlib.font_manager.findfont(_f, fallback_to_default=False)
        plt.rcParams["font.family"] = _f
        break
    except Exception:
        pass
plt.rcParams["axes.unicode_minus"] = False


def process(path, out_dir="output"):
    os.makedirs(out_dir, exist_ok=True)
    name = os.path.splitext(os.path.basename(path))[0]

    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(path)

    # 1) 노이즈 제거 -------------------------------------------------
    # 컬러 이미지용 Non-local Means (먼지/그레인 제거, 경계는 최대한 유지)
    den = cv2.fastNlMeansDenoisingColored(img, None, h=6, hColor=6,
                                          templateWindowSize=7, searchWindowSize=21)
    # 약한 가우시안 블러로 잔노이즈 한 번 더 정리
    smooth = cv2.GaussianBlur(den, (5, 5), 1.2)

    # 2) 가우시안 블러 이용한 핵 강조 (Unsharp mask) ------------------
    # 원본 + (원본 - 블러) * amount  -> 경계/핵이 또렷해짐
    blur_big = cv2.GaussianBlur(smooth, (0, 0), sigmaX=8)
    sharp = cv2.addWeighted(smooth, 1.5, blur_big, -0.5, 0)

    # 3) 색 변환: LAB에서 L채널 CLAHE로 대비 올리기 -------------------
    lab = cv2.cvtColor(sharp, cv2.COLOR_BGR2LAB)
    L, A, B = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    L2 = clahe.apply(L)
    enhanced = cv2.cvtColor(cv2.merge([L2, A, B]), cv2.COLOR_LAB2BGR)

    # 핵 구분용 시각화 1: HSV (핵은 채도 높고 명도 낮음)
    hsv = cv2.cvtColor(enhanced, cv2.COLOR_BGR2HSV)

    # 핵 구분용 시각화 2: 어두운 정도를 컬러맵으로
    # 핵은 주변(세포질)보다 훨씬 어두운 작은 덩어리 -> black-hat으로 추출
    gray = cv2.cvtColor(enhanced, cv2.COLOR_BGR2GRAY)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (41, 41))  # 핵보다 큰 커널
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, k)
    blackhat = cv2.normalize(blackhat, None, 0, 255, cv2.NORM_MINMAX)
    heat = cv2.applyColorMap(blackhat, cv2.COLORMAP_JET)

    # 색 변환 전/후 비교 (핵 검출 전) ---------------------------------
    rgb = lambda im: cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    panels = [
        (rgb(img),      "원본"),
        (rgb(sharp),    "색 변환 전 (노이즈 제거 + 언샤프)"),
        (rgb(enhanced), "색 변환 후 (LAB-CLAHE)"),
        (hsv[:, :, 1],  "HSV - 채도(S)"),
        (hsv[:, :, 2],  "HSV - 명도(V)"),
        (rgb(heat),     "히트맵 (black-hat)"),
    ]
    for ax, (im, title) in zip(axes.ravel(), panels):
        ax.imshow(im, cmap="gray" if im.ndim == 2 else None)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    fig.savefig(f"{out_dir}/{name}_0_compare.png", dpi=100)
    plt.show()

    # 4) 핵 분할 -----------------------------------------------------
    # 핵 = (a) 주변보다 국소적으로 어둡고 (b) 절대 밝기도 낮은 영역
    g = cv2.GaussianBlur(cv2.cvtColor(smooth, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    dark_thr = np.percentile(g, 4)                       # 이미지에서 가장 어두운 4%
    mask = ((g <= dark_thr) & (blackhat > 40)).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,
                            cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE,
                            cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)))

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    result = enhanced.copy()
    count = 0
    for c in contours:
        area = cv2.contourArea(c)
        if area < 150 or area > 6000:  # 이미지 배율에 맞게 조절
            continue
        perim = cv2.arcLength(c, True)
        circ = 4 * np.pi * area / (perim ** 2 + 1e-6)
        if circ < 0.6:  # 둥근 것만
            continue
        count += 1
        cv2.drawContours(result, [c], -1, (0, 255, 0), 2)
        x, y, w, h = cv2.boundingRect(c)
        cv2.putText(result, str(count), (x, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    print(f"[{name}] 검출된 핵: {count}개")

    # 저장
    cv2.imwrite(f"{out_dir}/{name}_1_denoised.png", smooth)
    cv2.imwrite(f"{out_dir}/{name}_2_sharpened.png", sharp)
    cv2.imwrite(f"{out_dir}/{name}_3_enhanced_clahe.png", enhanced)
    cv2.imwrite(f"{out_dir}/{name}_4_hsv.png", hsv)
    cv2.imwrite(f"{out_dir}/{name}_5_heatmap.png", heat)
    cv2.imwrite(f"{out_dir}/{name}_6_mask.png", mask)
    cv2.imwrite(f"{out_dir}/{name}_7_result.png", result)


if __name__ == "__main__":
    paths = [
        "image.png",
        "image2.png",
       
    ]
    for p in paths:
        process(p)

     